In [36]:
import numpy as np
import pandas as pd
import re

In [37]:
#import genotype data
genotype = pd.read_csv(
    "../data/raw/Pf8_drug_resistance_marker_genotypes.tsv",
    sep="\t"
)

In [38]:
# viewing data
genotype

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],kelch13_349-726_ns_changes,mdr1_dup_call,pm2_dup_call
0,FP0008-C,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
1,FP0009-C,C,I,E,T,CVIET,T,H,I,S,...,S,N,F,Y,VD,D,T,NaN,0,0
2,FP0010-CW,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
3,FP0011-CW,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
4,FP0012-CW,C,I,E,T,CVIET,T,H,I,S,...,S,N,F,D,VD,D,T,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,-,VD,D,T,-,-1,-1
24405,SPT92054,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,D,VD,D,T,NaN,0,-1
24406,SPT92057,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,D,VD,D,T,-,-1,-1
24407,SPT94772,C,I,E,T,CVIET,T,H,I,S,...,-,-,-,-,VD,D,-,-,-1,-1


In [39]:
# handling kelchi data
genotype['kelch13_349-726_ns_changes'].unique()

<StringArray>
[    nan,     '-', 'a676s', 'p441s', 'E612D', 'A676S', 'S522C', 'a578s',
 's522i', 'e509d',
 ...
 'v566i', 'p527s', 'a427v', 'v566l', 'd641n', 'S364Y', 'V487E', 's477y',
 'T350S', 'c469y']
Length: 266, dtype: str

In [40]:
genotype.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call'],
      dtype='str')

In [41]:
# viewing mdr duplication
genotype['mdr1_dup_call'].unique()

array([ 0, -1,  1])

In [42]:
genotype['mdr1_dup_call'].isna().sum()

np.int64(0)

In [43]:
genotype['pm2_dup_call'].unique()

array([ 0, -1,  1])

In [44]:
genotype['pm2_dup_call'].isna().sum()

np.int64(0)

In [45]:
# extract reference amino acid and create a dictionary

# Get all column names except Sample
cols = [c for c in genotype.columns if c != "Sample"]

# Dictionary: column -> reference amino acid
ref_aa = {}

for col in cols:
    m = re.search(r'\[(.*?)\]', col)
    if m:
        ref_aa[col] = m.group(1)

print(ref_aa)

{'crt_72[C]': 'C', 'crt_74[M]': 'M', 'crt_75[N]': 'N', 'crt_76[K]': 'K', 'crt_72-76[CVMNK]': 'CVMNK', 'crt_93[T]': 'T', 'crt_97[H]': 'H', 'crt_218[I]': 'I', 'crt_220[A]': 'A', 'crt_271[Q]': 'Q', 'crt_326[N]': 'N', 'crt_333[T]': 'T', 'crt_353[G]': 'G', 'crt_356[I]': 'I', 'crt_371[R]': 'R', 'dhfr_16[N]': 'N', 'dhfr_51[N]': 'N', 'dhfr_59[C]': 'C', 'dhfr_108[S]': 'S', 'dhfr_164[I]': 'I', 'dhfr_306[S]': 'S', 'dhps_436[S]': 'S', 'dhps_437[G]': 'G', 'dhps_540[K]': 'K', 'dhps_581[A]': 'A', 'dhps_613[A]': 'A', 'exo_415[E]': 'E', 'mdr1_86[N]': 'N', 'mdr1_184[Y]': 'Y', 'mdr1_1034[S]': 'S', 'mdr1_1042[N]': 'N', 'mdr1_1226[F]': 'F', 'mdr1_1246[D]': 'D', 'arps10_127-128[VD]': 'VD', 'fd_193[D]': 'D', 'mdr2_484[T]': 'T'}


In [46]:
import pandas as pd
import re

ref_df = pd.DataFrame({
    "column": genotype.columns
})

ref_df["reference"] = ref_df["column"].str.extract(r'\[(.*?)\]')

print(ref_df)

                        column reference
0                       Sample       NaN
1                    crt_72[C]         C
2                    crt_74[M]         M
3                    crt_75[N]         N
4                    crt_76[K]         K
5             crt_72-76[CVMNK]     CVMNK
6                    crt_93[T]         T
7                    crt_97[H]         H
8                   crt_218[I]         I
9                   crt_220[A]         A
10                  crt_271[Q]         Q
11                  crt_326[N]         N
12                  crt_333[T]         T
13                  crt_353[G]         G
14                  crt_356[I]         I
15                  crt_371[R]         R
16                  dhfr_16[N]         N
17                  dhfr_51[N]         N
18                  dhfr_59[C]         C
19                 dhfr_108[S]         S
20                 dhfr_164[I]         I
21                 dhfr_306[S]         S
22                 dhps_436[S]         S
23              

In [47]:
ref_df.dropna(inplace =True)

In [48]:
genotype.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call'],
      dtype='str')

In [49]:
ref_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 1 to 36
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   column     36 non-null     str  
 1   reference  36 non-null     str  
dtypes: str(2)
memory usage: 708.0 bytes


In [50]:
SPECIAL = {
    "crt_76[K]": {"T"},
    "dhfr_108[S]": {"N"},
    "dhps_437[G]": {"G"},   # using the Pf8 resistance allele
}

In [51]:
WHO_K13 = {
    "F446I","N458Y","M476I","Y493H","R539T","I543T",
    "P553L","R561H","P574L","C580Y",
    "P441L","G449A","C469F","C469Y","A481V","R515K",
    "P527H","N537I","N537D","G538V","V568G","R622I",
    "A675V","K479I","G533A","R575K","M579I",
    "D584V","P667T","F673I","H719N"
}

In [52]:
print(genotype["mdr1_dup_call"].unique())
print(genotype["pm2_dup_call"].unique())

[ 0 -1  1]
[ 0 -1  1]


In [53]:
def encode(value, column):

    # Missing
    if pd.isna(value) or str(value).strip() == "":
        return "Missing"

    value = str(value).strip()

    # ---------- Special: copy number ----------
    # ----- Special: copy number ----------
    if column in ["mdr1_dup_call", "pm2_dup_call"]:
        if int(value) == 0:
            return "Sensitive"
        elif int(value) == 1:
            return "Resistant"
        else:
            return "Undete"

    # ---------- Special: crt haplotype ----------
    if column == "crt_72-76[CVMNK]":

        haps = [x.strip() for x in value.split(",")]

        if any(h in {"CVIET", "SVMNT"} for h in haps):
            return "Resistant"

        if all(h == "CVMNK" for h in haps):
            return "Sensitive"

        return "Rare"

    # ---------- Special: kelch13 ----------
    if column == "kelch13_349-726_ns_changes":

        # Missing or unresolved
        if pd.isna(value):
            return "Rare"

        value = str(value).strip()

        if value in {"-", "*", "!"}:
            return "Rare"

        # Two haplotypes
        if "," in value:
            haps = [x.strip() for x in value.split(",")]

            # Same mutation on both haplotypes
            if len(set(haps)) == 1:
                value = haps[0]
            else:
                return "Rare"

        # Lowercase = heterozygous
        if value.islower():
            return "Rare"

        value = value.upper()

        # Wild type (if present in your data)
        if value in {"WT", "578S"}:
            return "Sensitive"

        # WHO resistance mutation
        if value in WHO_K13:
            return "Resistant"

        # Any other homozygous mutation
        return "Sensitive"
    
    

    # ---------- Multiple alleles ----------
    alleles = [a.strip() for a in value.split(",")]
    if len(alleles) > 1:

        if column in SPECIAL:
            if any(a in SPECIAL[column] for a in alleles):
                return "Resistant"
            return "Rare"

        # General columns
        if ref_aa[column] in alleles:
            return "Sensitive"

        return "Rare"

    # ---------- Single allele ----------

    allele = alleles[0]

    if column in SPECIAL:
        if allele in SPECIAL[column]:
            return "Resistant"

    if allele == ref_aa[column]:
        return "Sensitive"

    return "Resistant"

In [54]:
# columns for mutations in Sulfadoxine-Pyrimethamine (treatment) and Sulfadoxine-Pyrimethamine (IPTp)
dhfr_cols = [
    "dhfr_51[N]",
    "dhfr_59[C]",
    "dhfr_108[S]",
    "dhfr_164[I]"
]

dhps_cols = [
    "dhps_437[G]",
    "dhps_540[K]",
    "dhps_581[A]",
    "dhps_613[A]"
]

combo_cols = dhfr_cols + dhps_cols

In [55]:
# Encode only the independent markers
encoded = genotype.copy()

for col in genotype.columns:

    if col == "Sample":
        continue

    # Skip combination-rule loci
    if col in combo_cols:
        continue

    encoded[col] = genotype[col].apply(lambda x: encode(x, col))

In [56]:
# helper function
def is_ambiguous(value):

    if pd.isna(value):
        return True

    value = str(value).strip()

    if value in {"-", "*", "!"}:
        return True

    return value.islower()

In [57]:
# dealing with Pyrimethamine (triple mutant)
def classify_pyr(row):

    cols = [
        "dhfr_51[N]",
        "dhfr_59[C]",
        "dhfr_108[S]"
    ]

    if any(is_ambiguous(row[c]) for c in cols):
        return "Rare"

    if (
        row["dhfr_51[N]"] == "I"
        and row["dhfr_59[C]"] == "R"
        and row["dhfr_108[S]"] == "N"
    ):
        return "Resistant"

    return "Sensitive"

In [58]:
# dealing with SP-IPTp (sextuple mutant)
def classify_sp(row):

    cols = [
        "dhfr_51[N]",
        "dhfr_59[C]",
        "dhfr_108[S]",
        "dhfr_164[I]",
        "dhps_437[G]",
        "dhps_540[K]",
        "dhps_581[A]",
        "dhps_613[A]"
    ]

    if any(is_ambiguous(row[c]) for c in cols):
        return "Rare"

    if (
        row["dhfr_51[N]"] == "I"
        and row["dhfr_59[C]"] == "R"
        and row["dhfr_108[S]"] == "N"
        and row["dhps_437[G]"] == "G"
        and row["dhps_540[K]"] == "E"
        and (
            row["dhfr_164[I]"] == "L"
            or row["dhps_581[A]"] == "G"
            or row["dhps_613[A]"] in {"S", "T"}
        )
    ):
        return "Resistant"

    return "Sensitive"

In [59]:
# pyresistant is for triplet Pyrimethamine
encoded["PYRresistant"] = genotype.apply(classify_pyr, axis=1)
# Spresistant if for sextuple mutant
encoded["SPresistant"] = genotype.apply(classify_sp, axis=1)

In [60]:
encoded['PYRresistant'].value_counts()

PYRresistant
Resistant    17079
Sensitive     6712
Rare           618
Name: count, dtype: int64

In [61]:
encoded['SPresistant'].value_counts()

SPresistant
Sensitive    21176
Resistant     2088
Rare          1145
Name: count, dtype: int64

In [62]:
encoded.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call', 'PYRresistant', 'SPresistant'],
      dtype='str')

In [63]:
encoded[['kelch13_349-726_ns_changes','mdr1_dup_call']].head()

,kelch13_349-726_ns_changes,mdr1_dup_call
0,Missing,Sensitive
1,Missing,Sensitive
2,Missing,Sensitive
3,Missing,Sensitive
4,Missing,Sensitive


In [64]:
encoded['kelch13_349-726_ns_changes'].unique()

<StringArray>
['Missing', 'Rare', 'Sensitive', 'Resistant']
Length: 4, dtype: str

In [65]:
encoded['mdr1_dup_call'].unique()

<StringArray>
['Sensitive', 'Undete', 'Resistant']
Length: 3, dtype: str

### Handling artesunate_mefloquine

In [66]:
def classify_artesunate_mefloquine(row):

    art = row["kelch13_349-726_ns_changes"]
    mq = row["mdr1_dup_call"]

    # Missing
    if art == "Missing" or mq == "Missing":
        return "Missing"

    # Both resistant
    if art == "Resistant" and mq == "Resistant":
        return "Resistant"

    # At least one sensitive
    if art == "Sensitive" or mq == "Sensitive":
        return "Sensitive"

    # Rare/unknown combinations
    return "Rare"

In [67]:
encoded["ASMQresistant"] = encoded.apply(
    classify_artesunate_mefloquine,
    axis=1
)

In [68]:
encoded

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],kelch13_349-726_ns_changes,mdr1_dup_call,pm2_dup_call,PYRresistant,SPresistant,ASMQresistant
0,FP0008-C,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Sensitive,Sensitive,Missing
1,FP0009-C,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Resistant,...,Resistant,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
2,FP0010-CW,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
3,FP0011-CW,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
4,FP0012-CW,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Resistant,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Resistant,Sensitive,Sensitive,Sensitive,Rare,Undete,Undete,Resistant,Rare,Rare
24405,SPT92054,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Undete,Resistant,Sensitive,Missing
24406,SPT92057,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Rare,Undete,Undete,Resistant,Sensitive,Rare
24407,SPT94772,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Resistant,...,Resistant,Sensitive,Sensitive,Resistant,Rare,Undete,Undete,Resistant,Sensitive,Rare


## Filtering African countries only

In [69]:
# loading sample metadata
metadata = pd.read_csv(
    "../data/raw/Pf8_samples (1).txt",
    sep="\t",
    engine="python"
)

In [73]:
metadata

,Sample,Study,Country,Admin level 1,Country latitude,Country longitude,Admin level 1 latitude,Admin level 1 longitude,Year,ENA,All samples same case,Population,% callable,QC pass,Exclusion reason,Sample type,Sample was in Pf7
0,FP0008-C,1147-PF-MR-CONWAY,Mauritania,Hodh el Gharbi,20.265149,-10.337093,16.565426,-9.832345,2014.0,ERR1081237,FP0008-C,AF-W,82.48,True,Analysis_set,gDNA,True
1,FP0009-C,1147-PF-MR-CONWAY,Mauritania,Hodh el Gharbi,20.265149,-10.337093,16.565426,-9.832345,2014.0,ERR1081238,FP0009-C,AF-W,88.95,True,Analysis_set,gDNA,True
2,FP0010-CW,1147-PF-MR-CONWAY,Mauritania,Hodh el Gharbi,20.265149,-10.337093,16.565426,-9.832345,2014.0,ERR2889621,FP0010-CW,AF-W,87.01,True,Analysis_set,sWGA,True
3,FP0011-CW,1147-PF-MR-CONWAY,Mauritania,Hodh el Gharbi,20.265149,-10.337093,16.565426,-9.832345,2014.0,ERR2889624,FP0011-CW,AF-W,86.95,True,Analysis_set,sWGA,True
4,FP0012-CW,1147-PF-MR-CONWAY,Mauritania,Hodh el Gharbi,20.265149,-10.337093,16.565426,-9.832345,2014.0,ERR2889627,FP0012-CW,AF-W,89.86,True,Analysis_set,sWGA,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33320,SPT92268,1306-PF-NG-NGWA-SM,Nigeria,Oyo,9.592268,8.097575,8.143664,3.618846,2000.0,ERR10940681,SPT92268,AF-W,0.10,False,Low_coverage,sWGA,False
33321,SPT92269,1306-PF-NG-NGWA-SM,Nigeria,Oyo,9.592268,8.097575,8.143664,3.618846,2000.0,ERR11009733,SPT92269,AF-W,0.12,False,Low_coverage,sWGA,False
33322,SPT92270,1306-PF-NG-NGWA-SM,Nigeria,Oyo,9.592268,8.097575,8.143664,3.618846,2000.0,ERR11009737,SPT92270,AF-W,0.01,False,Low_coverage,sWGA,False
33323,SPT94772,1268-PF-MULTI-PAMGEN-SM,Gambia,Western,13.451482,-15.372910,13.245396,-16.401559,2017.0,ERR10789456,SPT94772,AF-W,64.96,True,Analysis_set,sWGA,False


In [74]:
metadata['Country'].unique()

<StringArray>
[                      'Mauritania',                           'Gambia',
                           'Guinea',                                nan,
                            'Kenya',                         'Thailand',
                         'Tanzania',                            'Ghana',
                         'Cambodia',                        'Indonesia',
                     'Burkina Faso',                             'Mali',
                 'Papua New Guinea',                             'Peru',
                       'Bangladesh',                           'Malawi',
                          'Vietnam',                         'Colombia',
                        'Venezuela',                           'Uganda',
                          'Myanmar',                             'Laos',
 'Democratic Republic of the Congo',                          'Nigeria',
                       'Madagascar',                         'Cameroon',
                    'Côte d'Ivoire', 

In [76]:
# filtering for African data
african_countries = [
    'Mauritania',
    'Gambia',
    'Guinea',
    'Kenya',
    'Tanzania',
    'Ghana',
    'Burkina Faso',
    'Mali',
    'Malawi',
    'Uganda',
    'Democratic Republic of the Congo',
    'Nigeria',
    'Madagascar',
    'Cameroon',
    "Côte d'Ivoire",
    'Ethiopia',
    'Benin',
    'Senegal',
    'Gabon',
    'Sudan',
    'Mozambique'
]

In [77]:
# african countries
africa_df = metadata[metadata['Country'].isin(african_countries)].copy()
africa_df.info()

<class 'pandas.DataFrame'>
Index: 20951 entries, 0 to 33324
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Sample                   20951 non-null  str    
 1   Study                    20951 non-null  str    
 2   Country                  20951 non-null  str    
 3   Admin level 1            20951 non-null  str    
 4   Country latitude         20951 non-null  float64
 5   Country longitude        20951 non-null  float64
 6   Admin level 1 latitude   20951 non-null  float64
 7   Admin level 1 longitude  20951 non-null  float64
 8   Year                     20951 non-null  float64
 9   ENA                      20951 non-null  str    
 10  All samples same case    20951 non-null  str    
 11  Population               20951 non-null  str    
 12  % callable               20951 non-null  float64
 13  QC pass                  20951 non-null  bool   
 14  Exclusion reason         20951 non-nul

In [79]:
africa_df.columns

Index(['Sample', 'Study', 'Country', 'Admin level 1', 'Country latitude',
       'Country longitude', 'Admin level 1 latitude',
       'Admin level 1 longitude', 'Year', 'ENA', 'All samples same case',
       'Population', '% callable', 'QC pass', 'Exclusion reason',
       'Sample type', 'Sample was in Pf7'],
      dtype='str')

In [81]:
# Merging with genotype encoded with africa df 
african_countries_geno = encoded.merge(africa_df, right_on = 'Sample',left_on= 'Sample', how = 'right')

In [82]:
print(len(african_countries_geno))

20951


In [83]:
african_countries_geno.head()

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,Admin level 1 longitude,Year,ENA,All samples same case,Population,% callable,QC pass,Exclusion reason,Sample type,Sample was in Pf7
0,FP0008-C,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,-9.832345,2014.0,ERR1081237,FP0008-C,AF-W,82.48,True,Analysis_set,gDNA,True
1,FP0009-C,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Resistant,...,-9.832345,2014.0,ERR1081238,FP0009-C,AF-W,88.95,True,Analysis_set,gDNA,True
2,FP0010-CW,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,-9.832345,2014.0,ERR2889621,FP0010-CW,AF-W,87.01,True,Analysis_set,sWGA,True
3,FP0011-CW,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,-9.832345,2014.0,ERR2889624,FP0011-CW,AF-W,86.95,True,Analysis_set,sWGA,True
4,FP0012-CW,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Resistant,...,-9.832345,2014.0,ERR2889627,FP0012-CW,AF-W,89.86,True,Analysis_set,sWGA,True


In [84]:
african_countries_geno.info()

<class 'pandas.DataFrame'>
RangeIndex: 20951 entries, 0 to 20950
Data columns (total 59 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Sample                      20951 non-null  str    
 1   crt_72[C]                   14508 non-null  str    
 2   crt_74[M]                   14508 non-null  str    
 3   crt_75[N]                   14508 non-null  str    
 4   crt_76[K]                   14508 non-null  str    
 5   crt_72-76[CVMNK]            14508 non-null  str    
 6   crt_93[T]                   14508 non-null  str    
 7   crt_97[H]                   14508 non-null  str    
 8   crt_218[I]                  14508 non-null  str    
 9   crt_220[A]                  14508 non-null  str    
 10  crt_271[Q]                  14508 non-null  str    
 11  crt_326[N]                  14508 non-null  str    
 12  crt_333[T]                  14508 non-null  str    
 13  crt_353[G]                  14508 non-null

In [85]:
# saving africa countries encoded genotype data
african_countries_geno.to_csv('../processed/africa_geno.csv')

# Merging with East African Countries

In [86]:
# loading east africa data 
east_genotype = pd.read_csv('../processed/east-country_year2010-2019.csv')

In [87]:
east_genotype.columns

Index(['Unnamed: 0', 'sample_id', 'year', 'qc_pass', 'study_id', 'region',
       'country', 'country_id', 'site', 'site_id', 'ARTresistant',
       'CQresistant', 'MQresistant', 'PPQresistant', 'PYRresistant',
       'SDXresistant'],
      dtype='str')

In [90]:
# merging east africa geno and african data
east_african_geno = east_genotype.merge(africa_df, right_on = 'Sample',left_on= 'sample_id', how = 'right')

In [91]:
# saving East africa countries encoded genotype data
east_african_geno.to_csv('../processed/East_africa_geno.csv')

# Merging with West African Countries

In [92]:
# loading wes africa data
west_genotype = pd.read_csv("../processed/west-country_year2010-2019.csv")

In [94]:
# merging east africa geno and african data
west_african_geno = west_genotype.merge(africa_df,right_on = 'Sample',left_on= 'sample_id', how = 'right')

In [95]:
# saving West africa countries encoded genotype data
west_african_geno.to_csv('../processed/West_africa_geno.csv')